# Notebook 5: Analyzing a Google ADK Agent Hierarchy

Import a Google Agent Development Kit (ADK) agent config and analyze
its delegation structure — transfers, sub-agents, handoffs.

**What this notebook shows:**
1. Define an ADK agent hierarchy (dict config or mock Agent objects)
2. Import with `from_adk_config()` — hierarchical IDs, role inference
3. Analyze the delegation DAG — where are the risks?
4. Compare flat vs hierarchical agent designs
5. Feed ADK session logs for calibration

In [ ]:
from google.adk import Agent

from minimal_oversight import analyze_pipeline
from minimal_oversight.connectors.adk import from_adk_config, from_adk_agent, normalize_adk_config, normalize_adk_agent
from minimal_oversight.connectors.traces import from_adk_session_logs, to_workflow_traces
from minimal_oversight.schema import NodeRole

import numpy as np

## 1. A real-world ADK agent config

**Scenario:** An e-commerce customer service system with specialized sub-agents.
The root agent triages, then delegates to order management, returns, or
product recommendations — each with their own sub-agents.

In [ ]:
ecommerce_config = {
    "name": "customer_service",
    "description": "Routes customer queries to the right specialist",
    "model": "gemini-2.0-flash",
    "sub_agents": [
        {
            "name": "order_manager",
            "description": "Handles order tracking, modifications, and cancellations",
            "model": "gemini-2.0-flash",
            "sub_agents": [
                {
                    "name": "order_lookup",
                    "description": "Searches order database and retrieves status",
                    "model": "gemini-2.0-flash",
                },
                {
                    "name": "order_modifier",
                    "description": "Writes order modifications (address, items)",
                    "model": "gemini-2.0-flash",
                },
            ],
        },
        {
            "name": "returns_handler",
            "description": "Processes return requests, checks eligibility, generates labels",
            "model": "gemini-2.0-flash",
        },
        {
            "name": "product_advisor",
            "description": "Recommends products based on preferences and history",
            "model": "gemini-2.0-flash",
            "sub_agents": [
                {
                    "name": "catalog_search",
                    "description": "Searches product catalog with filters",
                    "model": "gemini-2.0-flash",
                },
                {
                    "name": "review_analyzer",
                    "description": "Analyzes customer reviews for quality signals",
                    "model": "gemini-2.0-flash",
                },
            ],
        },
        {
            "name": "escalation",
            "description": "Escalates to human agent when confidence is low",
            "model": "gemini-2.0-flash",
        },
    ],
}

print("E-commerce customer service hierarchy:")
print("  customer_service (root)")
print("  ├── order_manager")
print("  │   ├── order_lookup")
print("  │   └── order_modifier")
print("  ├── returns_handler")
print("  ├── product_advisor")
print("  │   ├── catalog_search")
print("  │   └── review_analyzer")
print("  └── escalation")

In [ ]:
# Build with real ADK Agent objects
agent_root = Agent(
    name="customer_service",
    model="gemini-2.0-flash",
    description="Routes customer queries to the right specialist",
    instruction="You are a customer service router.",
    sub_agents=[
        Agent(
            name="order_manager",
            model="gemini-2.0-flash",
            description="Handles order tracking, modifications, and cancellations",
            sub_agents=[
                Agent(name="order_lookup", model="gemini-2.0-flash",
                      description="Searches order database and retrieves status"),
                Agent(name="order_modifier", model="gemini-2.0-flash",
                      description="Writes order modifications (address, items)"),
            ],
        ),
        Agent(name="returns_handler", model="gemini-2.0-flash",
              description="Processes return requests"),
        Agent(
            name="product_advisor",
            model="gemini-2.0-flash",
            description="Recommends products based on preferences",
            sub_agents=[
                Agent(name="catalog_search", model="gemini-2.0-flash",
                      description="Searches product catalog with filters"),
                Agent(name="review_analyzer", model="gemini-2.0-flash",
                      description="Analyzes customer reviews for quality signals"),
            ],
        ),
        Agent(name="escalation", model="gemini-2.0-flash",
              description="Escalates to human agent when confidence is low"),
    ],
)

# Import real Agent object
agent_pipeline = from_adk_agent(agent_root)
print(f"From Agent object: {agent_pipeline}")
print(f"Depth: {agent_pipeline.depth}")

# Auto-detect works on real Agent objects
agent_report = analyze_pipeline(agent_root, p_min=0.70)
print(f"Auto-detect on Agent: C_op={agent_report.feasibility.c_op:.3f}")

# Compare: dict config produces the same result
config_pipeline = from_adk_config(ecommerce_config)
config_report = analyze_pipeline(config_pipeline, p_min=0.70)
print(f"From dict config:     C_op={config_report.feasibility.c_op:.3f}")
print(f"Match: {np.isclose(agent_report.feasibility.c_op, config_report.feasibility.c_op)}")

### Using real ADK Agent objects

The same hierarchy can be built with real `google.adk.Agent` objects.
Both `from_adk_agent()` and `from_adk_config()` produce the same pipeline.

## 2. Import and inspect

In [ ]:
# See what the normalizer infers
normalized = normalize_adk_config(ecommerce_config)

print(f"{'ID':<45} {'Role':<12} {'Description'}")
print("-" * 100)
for n in normalized.nodes:
    desc = n.description[:50] + "..." if len(n.description) > 50 else n.description
    print(f"{n.id:<45} {n.role.value:<12} {desc}")

print(f"\nEdges:")
for e in normalized.edges:
    handoff = " [handoff]" if e.is_handoff else ""
    print(f"  {e.source_id} → {e.target_id}{handoff}")

In [ ]:
# Convert and analyze
pipeline = from_adk_config(ecommerce_config)
print(pipeline)
print(f"Depth: {pipeline.depth}")

## 3. Full analysis

In [ ]:
report = analyze_pipeline(pipeline, p_min=0.70)
print(report)

## 4. Auto-detect: pass the dict directly

`analyze_pipeline()` auto-detects ADK configs when the dict has `sub_agents`.

In [ ]:
# Auto-detect from dict
auto_report = analyze_pipeline(ecommerce_config, p_min=0.70)
print(f"Auto-detected: C_op={auto_report.feasibility.c_op:.3f}")
print(f"Feasible: {auto_report.feasibility.feasible}")

## 5. Where are the risks?

The root agent is a fan-out node — it delegates to 4 sub-agents.
Its delegation centrality is highest because a failure there
propagates to every branch.

In [ ]:
print(f"{'Node':<45} {'DC':>6} {'M*':>6} {'SOTA':>8} {'Fan-out':>8}")
print("-" * 78)
for risk in report.node_risks:
    m = f"{risk.masking_index:.2f}" if risk.masking_index else "  n/a"
    s = f"{risk.sota_score:.3f}" if risk.sota_score else "   n/a"
    print(f"{risk.name:<45} {risk.delegation_centrality:>6.1f} {m:>6} {s:>8} {risk.fan_out_degree:>8}")

## 6. Flat vs hierarchical design

What if we flatten the hierarchy? Instead of root → order_manager → {lookup, modifier},
make root delegate directly to all leaf agents.

In [ ]:
flat_config = {
    "name": "customer_service_flat",
    "description": "Routes to all specialists directly",
    "model": "gemini-2.0-flash",
    "sub_agents": [
        {"name": "order_lookup", "description": "Searches order database"},
        {"name": "order_modifier", "description": "Writes order modifications"},
        {"name": "returns_handler", "description": "Processes returns"},
        {"name": "catalog_search", "description": "Searches product catalog"},
        {"name": "review_analyzer", "description": "Analyzes reviews"},
        {"name": "escalation", "description": "Escalates to human"},
    ],
}

flat_pipeline = from_adk_config(flat_config)
flat_report = analyze_pipeline(flat_pipeline, p_min=0.70)

print("=== Hierarchical vs Flat ===")
print(f"\n{'Metric':<30} {'Hierarchical':>14} {'Flat':>14}")
print("-" * 60)
print(f"{'Nodes':<30} {len(pipeline.nodes):>14} {len(flat_pipeline.nodes):>14}")
print(f"{'Depth':<30} {pipeline.depth:>14} {flat_pipeline.depth:>14}")
print(f"{'C_op':<30} {report.feasibility.c_op:>14.3f} {flat_report.feasibility.c_op:>14.3f}")
print(f"{'B_eff':<30} {report.feasibility.b_eff:>14.4f} {flat_report.feasibility.b_eff:>14.4f}")
print(f"{'Bottleneck':<30} {report.feasibility.bottleneck_node or '-':>14} {flat_report.feasibility.bottleneck_node or '-':>14}")
print(f"{'Motifs detected':<30} {len(report.motifs):>14} {len(flat_report.motifs):>14}")

print("\nFlat is shallower (depth 1) → higher C_op, less masking accumulation.")
print("But flat has higher fan-out → more cascade risk from the root.")
print("Trade-off: depth costs quality, fan-out costs resilience.")

## 7. Feed ADK session logs

Parse real session events to replace heuristic defaults with measured performance.

In [ ]:
# Simulated ADK session logs
# In production: sessions = adk_client.list_sessions(agent_id="customer_service")
rng = np.random.default_rng(42)
simulated_sessions = []

# Use flat agent names matching how ADK logs events
agent_skills = {"customer_service": 0.75, "order_manager": 0.60,
                "order_lookup": 0.80, "returns_handler": 0.55}

for i in range(300):
    events = []
    # Root triage — always a transfer (not a processing event)
    events.append({
        "agent": "customer_service",
        "type": "transfer",
        "timestamp": f"2026-03-{20 + i // 100:02d}T{10 + (i % 10)}:00:00Z",
    })
    # Delegated agent processes
    chosen = rng.choice(["order_manager", "returns_handler"])
    success = rng.random() < agent_skills[chosen]
    events.append({
        "agent": chosen,
        "type": "response" if success else "error",
        "timestamp": f"2026-03-{20 + i // 100:02d}T{10 + (i % 10)}:01:00Z",
    })
    # Sub-agent if order_manager
    if chosen == "order_manager":
        lookup_ok = rng.random() < agent_skills["order_lookup"]
        events.append({
            "agent": "order_lookup",
            "type": "response" if lookup_ok else "error",
            "timestamp": f"2026-03-{20 + i // 100:02d}T{10 + (i % 10)}:01:30Z",
        })
    simulated_sessions.append({"id": f"session_{i:04d}", "events": events})

print(f"Simulated {len(simulated_sessions)} ADK sessions.")

In [ ]:
# Parse session logs into normalized traces
norm_traces = from_adk_session_logs(simulated_sessions)
wf_traces = to_workflow_traces(norm_traces)
print(f"Parsed {len(norm_traces)} sessions → {len(wf_traces)} workflow traces")

# For calibration, use a flat pipeline so node IDs match the trace agent names.
# (The hierarchical pipeline uses IDs like "customer_service/order_manager",
# but ADK session logs use flat names like "order_manager".)
from minimal_oversight.models import Node, PipelineGraph

cal_nodes = [
    Node("customer_service", sigma_skill=0.55, catch_rate=0.65, review_capacity=0.50),
    Node("order_manager", sigma_skill=0.55, catch_rate=0.65, review_capacity=0.50),
    Node("order_lookup", sigma_skill=0.55, catch_rate=0.65, review_capacity=0.50),
    Node("returns_handler", sigma_skill=0.55, catch_rate=0.65, review_capacity=0.50),
]
cal_pipeline = PipelineGraph(cal_nodes)
cal_pipeline.add_edge("customer_service", "order_manager")
cal_pipeline.add_edge("customer_service", "returns_handler")
cal_pipeline.add_edge("order_manager", "order_lookup")

cal_report = analyze_pipeline(cal_pipeline, p_min=0.70, traces=wf_traces)

print(f"\nCalibrated C_op: {cal_report.feasibility.c_op:.3f}")
print(f"Feasible: {cal_report.feasibility.feasible}")
print(f"\nEstimated node performance (from {len(wf_traces)} sessions):")
for name, est in cal_report.node_estimates.items():
    if est.sample_size > 0:
        print(f"  {name}: σ_raw={est.sigma_raw:.3f} σ_corr={est.sigma_corr:.3f} M*={est.masking_index:.2f} (n={est.sample_size})")
    else:
        print(f"  {name}: no observations in traces")

## 8. Where should the SOTA model go?

If you have budget for one Gemini Pro call per request, where does it go?
The theory predicts: the node with highest delegation centrality benefits most.

In [ ]:
print("SOTA placement test: upgrade one node at a time to σ_skill=0.90\n")
print(f"{'Upgraded node':<45} {'C_op':>8} {'Δ':>8}")
print("-" * 65)

baseline = from_adk_config(ecommerce_config)
base_report = analyze_pipeline(baseline, p_min=0.70)
base_c = base_report.feasibility.c_op
print(f"{'(baseline)':<45} {base_c:>8.3f} {0:>+8.3f}")

best_node = None
best_delta = 0.0

for node_id in pipeline.topological_order():
    p = from_adk_config(ecommerce_config)
    p.get_node(node_id).sigma_skill = 0.90
    r = analyze_pipeline(p, p_min=0.70)
    delta = r.feasibility.c_op - base_c
    if delta > best_delta:
        best_delta = delta
        best_node = node_id
    print(f"{node_id:<45} {r.feasibility.c_op:>8.3f} {delta:>+8.4f}")

print(f"\nBest placement: {best_node} (Δ={best_delta:+.4f})")
print("The node with highest delegation centrality benefits most from upgrade.")

## 9. Failure surface

In [ ]:
explanation = report.failure_explanation.strip()
if explanation:
    print(explanation)
else:
    print("No significant failure modes detected. Pipeline is operating within safe margins.")

## Key takeaways

1. **ADK hierarchies map directly to delegation DAGs** — the connector walks `sub_agents` recursively and builds the graph with hierarchical IDs (`root/parent/child`).
2. **Root agent = fan-out node** — its delegation centrality is highest because every sub-agent depends on correct routing.
3. **Flat vs hierarchical is a real design tradeoff** — flat has less depth (higher C_op) but more fan-out (more cascade risk). The theory quantifies both.
4. **Session logs calibrate the model** — replace heuristic defaults with measured success rates from ADK session events.
5. **SOTA placement follows delegation centrality** — upgrade the node whose correction propagates farthest, not the one with the lowest raw skill.